In [ ]:
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import cartopy.crs as ccrs
import cartopy.feature as cfeature
from cartopy.mpl.gridliner import (LongitudeFormatter,
                                   LatitudeFormatter)
from whakaaribn import get_data

In [ ]:
fn = get_data('data/271912_Brad Scott_GNS Science.jpg')
arr_img = plt.imread(fn, format='jpg')
y, x, z = arr_img.shape
factor=300
fig = plt.figure(figsize=(8, 6))
ax1 = fig.add_axes([0.02,0.525,0.5,0.45], projection=ccrs.Mercator())
ax2 = fig.add_axes([0.48, 0.5, 0.51, 0.5])
ax3 = fig.add_axes([0.1,0.05,0.4,0.4])
ax4 = fig.add_axes([0.55,0.05,0.4,0.4])


# Add a Map
ax1.set_extent([160, 180, -49, -32], crs=ccrs.PlateCarree())
ax1.coastlines(resolution='50m')
ax1.add_feature(cfeature.LAND, facecolor='lightgray')
ax1.add_feature(cfeature.OCEAN, facecolor='lightblue')
url = "https://basemaps.linz.govt.nz/v1/tiles/aerial/WebMercatorQuad/WMTSCapabilities.xml?api=c01jj05fc72acjxevhtrem76m80"
layer = "aerial"
ax1.add_wmts(url, layer)

label_style = {'color': 'black', 'weight': 'bold', 'size': 8}
gl = ax1.gridlines(crs=ccrs.PlateCarree(), draw_labels=True,
                                     linewidth=1, color='gray', alpha=0.5,
                                     linestyle='--', xlabel_style=label_style,
                                     ylabel_style=label_style)
gl.top_labels = False
gl.right_labels = False
gl.xlines = True
gl.ylines = True
gl.xlocator = mticker.FixedLocator([166, 172])
gl.ylocator = mticker.FixedLocator([-45, -40, -35])
gl.xformatter = LongitudeFormatter(direction_label=True)
gl.yformatter = LatitudeFormatter(direction_label=True)
gl.xpadding = -1
gl.ypadding = -5
ax1.plot(177.18270881065772, -37.51992819737241, marker='o', color='red',
                  markersize=3, transform=ccrs.PlateCarree())
ax1.text(
    .1,
    .9,
    "(A)",
    horizontalalignment="center",
    transform=ax1.transAxes,
    fontdict={"color": "k", "fontsize": 14}
)

# Add an image of Whakaari
ax2.imshow(arr_img)
ax2.set_xticks([])
ax2.set_yticks([])
ax2.text(
    .1,
    .9,
    "(B)",
    horizontalalignment="center",
    transform=ax2.transAxes,
    fontdict={"color": "k", "fontsize": 14}
)

# Add Network structure for causal model
magma_edges = [('Magmatic Intrusion', 'Eruption'),
               ('Magmatic Intrusion', 'Eqr'),
               ('Magmatic Intrusion', 'RSAM'),
               ('Magmatic Intrusion', 'SO2'),
               ('Magmatic Intrusion', 'CO2'),
               ('Magmatic Intrusion', 'H2S')]
seal_edges = [('Hydrothermal Seal', 'Eruption'),
              ('Hydrothermal Seal', 'Eqr'),
              ('Hydrothermal Seal', 'RSAM'),
              ('Hydrothermal Seal', 'SO2'),
              ('Hydrothermal Seal', 'CO2'),
              ('Hydrothermal Seal', 'H2S')]

G1 = nx.DiGraph(magma_edges + seal_edges)

G2 = nx.DiGraph([('Eruption', 'Eqr'),
                 ('Eruption', 'RSAM'),
                 ('Eruption', 'SO2'),
                 ('Eruption', 'CO2'),
                 ('Eruption', 'H2S'),
                 ('Eqr', 'RSAM'),
                 ('Eqr', 'SO2'),
                 ('Eqr', 'CO2'),
                 ('Eqr', 'H2S'),
                 ('RSAM', 'SO2'),
                 ('RSAM', 'CO2'),
                 ('RSAM', 'H2S'),
                 ('CO2', 'SO2'),
                 ('CO2', 'H2S'),
                 ('SO2', 'H2S')])

# group nodes by rows
bottom_nodes = ['Magmatic Intrusion', 'Hydrothermal Seal']
top_nodes = ['Eqr', 'RSAM', 'SO2', 'CO2', 'H2S']

# set the position according to column (x-coord)
x_pos_bottom = [0., 1.1]
x_pos_top = [0., 0.35, 0.7, 1, 1.3]
pos = {n: (_x, 0) for n, _x in zip(bottom_nodes, x_pos_bottom)}
pos.update({n: (_x, .5) for n, _x in zip(top_nodes, x_pos_top)})
pos.update({'Eruption': (-.3, 0.25)})
pos = nx.spring_layout(G1, pos=pos)
pos.update({'Magmatic Intrusion': (-.3, 0.25)})
pos.update({'Hydrothermal Seal': (.3, -0.25)})

arrowsize = 14
nx.draw_networkx_edges(G1, pos, edgelist=magma_edges,
                       edge_color='r', ax= ax3,
                       arrowsize=arrowsize)
nx.draw_networkx_edges(G1, pos, edgelist=seal_edges,
                       edge_color='b', ax= ax3,
                       arrowsize=arrowsize)

label_options = {"fc": "white", "alpha": 0.5}
nx.draw_networkx_labels(G1, pos, font_size=12, ax=ax3, bbox=label_options, verticalalignment='center')
nx.draw_networkx_nodes(G1, pos, ax= ax3)

ax3.axis("off")
ax3.text(
    0.1,
    0.9,
    "(C)",
    horizontalalignment="center",
    transform=ax3.transAxes,
    fontdict={"color": "k", "fontsize": 14}
)

# Add network structure for fully-connected model
pos = nx.kamada_kawai_layout(G2)
nx.draw_networkx_edges(G2, pos, ax= ax4, arrowsize=arrowsize)
nx.draw_networkx_nodes(G2, pos, ax= ax4)
label_options = {"fc": "white", "alpha": 0.5}
nx.draw_networkx_labels(G2, pos, font_size=12, ax=ax4, bbox=label_options, verticalalignment='center')
ax4.set_xlim(-1.2, 1.35)
ax4.axis("off")
ax4.text(
    0.05,
    0.9,
    "(D)",
    horizontalalignment="center",
    transform=ax4.transAxes,
    fontdict={"color": "k", "fontsize": 14}
)
try:
    fig.savefig(snakemake.output[0], dpi=300, bbox_inches='tight')
except NameError:
    pass
fig
